# 🚀 Essential ML Coding Interview Guide for MLEs
### From-Scratch Implementations in NumPy & PyTorch

This notebook is an end-to-end, interview-focused masterclass covering the **most commonly asked Machine Learning coding questions** in top-tier Machine Learning Engineer (MLE) and Applied Scientist interviews (Meta, OpenAI, Anthropic, Apple, Amazon, Databricks).

In these interviews, candidates are evaluated on:
1. **Mathematical rigor**: Deriving gradients from first principles.
2. **Vectorization**: Avoiding nested loops and mastering tensor broadcasting.
3. **Numerical stability**: Preventing underflow/overflow (log-sum-exp, stable sigmoid, epsilon offsets).
4. **Deep learning architectures**: Building core building blocks (`nn.Module`, parameter initialization traps, tensor shape transformations).

---

## 📑 Table of Contents
1. [Linear Regression (Gradient Descent & Normal Equation)](#1-linear-regression)
2. [Logistic Regression (Gradient Descent & Numerical Stability)](#2-logistic-regression)
3. [K-Means Clustering (Random Choice & List Comprehension)](#3-k-means-clustering)
4. [K-Nearest Neighbors (KNN - Unified Distance & Top-K)](#4-k-nearest-neighbors-knn)
5. [2-Layer MLP with Backpropagation from Scratch (NumPy)](#5-2-layer-mlp-with-backpropagation-from-scratch)
6. [Layer Normalization (LayerNorm in NumPy & PyTorch)](#6-layer-normalization-layernorm)
7. [Multi-Head Self-Attention (Transformer Core)](#7-multi-head-self-attention)
8. [Low-Rank Adaptation (LoRA) Layer](#8-low-rank-adaptation-lora)
9. [Byte-Pair Encoding (BPE) Tokenizer](#9-byte-pair-encoding-bpe)
10. [MLE Interview Quick Reference & Cheat Sheet](#10-mle-interview-quick-reference--cheat-sheet)


---
## 1. Linear Regression

Linear Regression models a linear relationship between input features $X \in \mathbb{R}^{N \times D}$ and continuous targets $y \in \mathbb{R}^{N}$.

### Mathematical Formulation
$$
\hat{y} = X w + b
$$

We optimize the Mean Squared Error (MSE) objective:
$$
\mathcal{L}(w, b) = \frac{1}{2N} \sum_{i=1}^N (\hat{y}_i - y_i)^2 = \frac{1}{2N} \|X w + b - y\|_2^2
$$

*(The $\frac{1}{2}$ prefactor simplifies the derivative by canceling with the exponent 2).*

### Gradients Derivation
Taking partial derivatives with respect to weights $w$ and bias $b$:
$$
\frac{\partial \mathcal{L}}{\partial w} = \frac{1}{N} X^T (\hat{y} - y) \in \mathbb{R}^{D}
$$
$$
\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i) = \frac{1}{N} \mathbf{1}^T (\hat{y} - y) \in \mathbb{R}
$$

### Closed-Form Analytical Solution (Normal Equation)
By augmenting $X$ with an intercept column of ones $\tilde{X} = [X, \mathbf{1}] \in \mathbb{R}^{N \times (D+1)}$ and setting $\nabla_{\tilde{w}} \mathcal{L} = 0$:
$$
\tilde{w} = (\tilde{X}^T \tilde{X})^{-1} \tilde{X}^T y
$$

> **💡 Interview Gotchas & Pro-Tips:**
> - **Invertibility trap:** If features are collinear (or $D > N$), $\tilde{X}^T \tilde{X}$ is singular (non-invertible). Always mention or use the Moore-Penrose pseudo-inverse (`np.linalg.pinv`), which computes the minimum-norm solution via SVD.
> - **Feature Scaling:** Gradient descent requires feature normalization ($z$-score standardizing $X$) to ensure spherical loss contours and fast convergence. The Normal Equation does not require feature scaling.
> - **Complexity:** Normal Equation is $\mathcal{O}(D^3 + N D^2)$ time and $\mathcal{O}(D^2)$ memory. Gradient Descent is $\mathcal{O}(\text{epochs} \cdot N D)$. For large $D$ ($D > 10^4$), GD is strongly preferred.


In [ ]:
import numpy as np

class LinearRegressionGD:
    """
    Linear Regression implemented from scratch with NumPy.
    Supports both iterative Batch Gradient Descent and the analytical Normal Equation.
    """
    def __init__(self, lr: float = 0.01, n_iters: int = 1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LinearRegressionGD":
        """
        Fit weights using vectorized batch gradient descent.
        X: shape (N, D)
        y: shape (N,) or (N, 1)
        """
        N, D = X.shape
        y = y.reshape(-1, 1)
        
        # Initialize parameters: zeros
        self.weights = np.zeros((D, 1))
        self.bias = 0.0
        self.loss_history = []

        for _ in range(self.n_iters):
            # 1. Forward prediction
            y_pred = np.dot(X, self.weights) + self.bias  # Shape: (N, 1)
            
            # 2. Compute error and MSE loss
            error = y_pred - y                            # Shape: (N, 1)
            loss = (1.0 / (2.0 * N)) * np.sum(error ** 2)
            self.loss_history.append(loss)
            
            # 3. Vectorized gradients
            dw = (1.0 / N) * np.dot(X.T, error)          # Shape: (D, 1)
            db = (1.0 / N) * np.sum(error)               # Scalar
            
            # 4. Parameter update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict continuous values for input X."""
        return (np.dot(X, self.weights) + self.bias).squeeze()

    def fit_normal_equation(self, X: np.ndarray, y: np.ndarray) -> "LinearRegressionGD":
        """
        Analytical solution: w_aug = (X_aug^T X_aug)^(-1) X_aug^T y
        Uses pseudo-inverse for numerical stability with singular matrices.
        """
        N, D = X.shape
        y = y.reshape(-1, 1)
        
        # Augment X with a column of 1s for the bias term
        X_aug = np.hstack([X, np.ones((N, 1))])  # Shape: (N, D + 1)
        
        # Compute pseudo-inverse to handle collinearity gracefully
        w_aug = np.linalg.pinv(X_aug.T @ X_aug) @ (X_aug.T @ y)
        
        self.weights = w_aug[:-1]
        self.bias = float(w_aug[-1, 0])
        return self


In [ ]:
# --- Verification & Unit Test: Linear Regression ---
np.random.seed(42)
N, D = 200, 3

# True parameters: y = 3.5*x0 - 2.0*x1 + 1.2*x2 + 4.0 + noise
true_w = np.array([[3.5], [-2.0], [1.2]])
true_b = 4.0

X = np.random.randn(N, D)
noise = np.random.randn(N, 1) * 0.05
y = X @ true_w + true_b + noise

# 1. Fit with Gradient Descent
model_gd = LinearRegressionGD(lr=0.05, n_iters=1500)
model_gd.fit(X, y)

# 2. Fit with Normal Equation
model_ne = LinearRegressionGD()
model_ne.fit_normal_equation(X, y)

print("Linear Regression Results:")
print(f"True Weights: {true_w.ravel()}, True Bias: {true_b:.4f}")
print(f"GD Weights:   {model_gd.weights.ravel()}, GD Bias:   {model_gd.bias:.4f}")
print(f"NE Weights:   {model_ne.weights.ravel()}, NE Bias:   {model_ne.bias:.4f}")

# Check that GD converged close to Normal Equation
assert np.allclose(model_gd.weights, model_ne.weights, atol=1e-2), "GD weights did not match Normal Equation!"
assert np.isclose(model_gd.bias, model_ne.bias, atol=1e-2), "GD bias did not match Normal Equation!"
print("✅ Linear Regression passed all verification checks!")


---
## 2. Logistic Regression

Logistic Regression is the foundational classification model that maps linear combinations of features to class probabilities $\hat{y} \in (0, 1)$ via the Sigmoid activation function.

### Mathematical Formulation
$$
\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}, \quad \text{where } z = X w + b
$$

The objective is to minimize the **Binary Cross-Entropy (BCE) Loss**:
$$
\mathcal{L}(w, b) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]
$$

### Gradients Derivation
Using the chain rule and the identity $\frac{d\sigma(z)}{dz} = \sigma(z)(1 - \sigma(z))$:
$$
\frac{\partial \mathcal{L}}{\partial \hat{y}_i} = -\frac{y_i}{\hat{y}_i} + \frac{1 - y_i}{1 - \hat{y}_i} = \frac{\hat{y}_i - y_i}{\hat{y}_i(1 - \hat{y}_i)}
$$
$$
\frac{\partial \hat{y}_i}{\partial z_i} = \hat{y}_i(1 - \hat{y}_i)
$$
Multiplying gives:
$$
\frac{\partial \mathcal{L}}{\partial z_i} = \hat{y}_i - y_i
$$
Vectorized across all samples:
$$
\frac{\partial \mathcal{L}}{\partial w} = \frac{1}{N} X^T (\hat{y} - y) \in \mathbb{R}^D
$$
$$
\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i) \in \mathbb{R}
$$

> **💡 Interview Gotchas & Pro-Tips:**
> - **Sigmoid Overflow / Underflow:** When $z \ll 0$, $e^{-z} \to \infty$ which raises `OverflowError`. A numerically stable sigmoid branches based on sign:
>   $$
>   \sigma(z) = \begin{cases} \frac{1}{1 + e^{-z}} & \text{if } z \ge 0 \\ \frac{e^z}{1 + e^z} & \text{if } z < 0 \end{cases}
>   $$
> - **Log(0) Explosion:** If $\hat{y} = 0$, $\log(\hat{y}) = -\infty$. Always clip probabilities: `np.clip(y_pred, 1e-15, 1 - 1e-15)`.


In [ ]:
class LogisticRegressionGD:
    """
    Logistic Regression with numerical stability tricks implemented from scratch in NumPy.
    """
    def __init__(self, lr: float = 0.1, n_iters: int = 1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.loss_history = []

    @staticmethod
    def _stable_sigmoid(z: np.ndarray) -> np.ndarray:
        """
        Numerically stable sigmoid function:
        For z >= 0: 1 / (1 + exp(-z))
        For z < 0:  exp(z) / (1 + exp(z))
        """
        z = np.asarray(z, dtype=np.float64)
        positive_mask = (z >= 0)
        negative_mask = ~positive_mask
        
        result = np.empty_like(z)
        result[positive_mask] = 1.0 / (1.0 + np.exp(-z[positive_mask]))
        exp_z_neg = np.exp(z[negative_mask])
        result[negative_mask] = exp_z_neg / (1.0 + exp_z_neg)
        return result

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LogisticRegressionGD":
        N, D = X.shape
        y = y.reshape(-1, 1)
        
        self.weights = np.zeros((D, 1))
        self.bias = 0.0
        self.loss_history = []

        eps = 1e-15  # For stable log computation

        for _ in range(self.n_iters):
            # 1. Forward linear combination + sigmoid
            linear_output = np.dot(X, self.weights) + self.bias
            y_pred = self._stable_sigmoid(linear_output)
            
            # 2. Binary Cross-Entropy loss with numerical clipping
            y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
            loss = - (1.0 / N) * np.sum(
                y * np.log(y_pred_clipped) + (1.0 - y) * np.log(1.0 - y_pred_clipped)
            )
            self.loss_history.append(loss)
            
            # 3. Vectorized gradients
            error = y_pred - y
            dw = (1.0 / N) * np.dot(X.T, error)
            db = (1.0 / N) * np.sum(error)
            
            # 4. Gradient descent update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return predicted probability for class 1."""
        linear_output = np.dot(X, self.weights) + self.bias
        return self._stable_sigmoid(linear_output).squeeze()

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """Return binary class label (0 or 1)."""
        return (self.predict_proba(X) >= threshold).astype(int)


In [ ]:
# --- Verification & Unit Test: Logistic Regression ---
np.random.seed(42)
N = 300

# Create two clearly separable 2D Gaussian clusters
X0 = np.random.randn(N // 2, 2) + np.array([-2.0, -2.0])
X1 = np.random.randn(N // 2, 2) + np.array([2.0, 2.0])
X = np.vstack([X0, X1])
y = np.array([0] * (N // 2) + [1] * (N // 2))

# Train model
clf = LogisticRegressionGD(lr=0.1, n_iters=1000)
clf.fit(X, y)

preds = clf.predict(X)
accuracy = np.mean(preds == y)

print(f"Trained Logistic Regression Accuracy: {accuracy * 100:.2f}%")
print(f"Initial Loss: {clf.loss_history[0]:.4f} -> Final Loss: {clf.loss_history[-1]:.4f}")

assert accuracy > 0.95, f"Expected >95% accuracy on separable data, got {accuracy * 100}%"
assert clf.loss_history[-1] < clf.loss_history[0], "Loss failed to decrease during training!"
print("✅ Logistic Regression passed all verification checks!")


---
## 3. K-Means Clustering

K-Means partitions $N$ observations into $K$ clusters such that each observation belongs to the cluster with the nearest mean (centroid).

### Mathematical Objective
Minimize within-cluster sum of squared distances:
$$
\mathcal{J} = \sum_{k=1}^K \sum_{x \in \mathcal{C}_k} \|x - \mu_k\|_2^2
$$

### Algorithm Steps in a Single Unified Function
1. **Centroid Initialization:** Randomly choose $K$ distinct data points from $X$ without replacement:
   $$
   \text{indices} = \text{random\_choice}(N, K, \text{replace}=\text{False})
   $$
2. **Assignment Step (Broadcasting):** Assign each sample $x_i$ to its nearest centroid:
   $$
   c_i = \arg\min_{k \in \{1, \dots, K\}} \|x_i - \mu_k\|_2^2
   $$
3. **Update Step (List Comprehension):** Recalculate centroids as the mean of assigned points using a Python list comprehension:
   $$
   \mu_k = \frac{1}{|\mathcal{C}_k|} \sum_{x \in \mathcal{C}_k} x
   $$
4. **Convergence:** Terminate when the maximum centroid shift across iterations is less than tolerance $\text{tol}$ or when $\text{max\_iters}$ is reached.

> **💡 Interview Gotchas & Pro-Tips:**
> - **Unified Function:** Keep initialization and the iteration loop concise within a single `fit()` function.
> - **Pythonic List Comprehension:** Interviewers appreciate concise vectorization: updating centroids with `np.array([np.mean(X[labels == k], axis=0) for k in range(self.k)])`.
> - **Vectorized Broadcasting:** Pairwise distances are computed in one line `np.linalg.norm(X[:, None, :] - self.centroids[None, :, :], axis=2)` without nested loops.


In [ ]:
class KMeans:
    """
    K-Means clustering implemented with a single unified fit() function.
    Uses random choice initialization, vectorized distance broadcasting,
    and list comprehension for centroid updates.
    """
    def __init__(self, k: int = 3, max_iters: int = 300, tol: float = 1e-4):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.centroids = None

    def fit(self, X: np.ndarray) -> "KMeans":
        N, D = X.shape

        # 1. Random centroid initialization without replacement
        random_indices = np.random.choice(N, size=self.k, replace=False)
        self.centroids = X[random_indices].astype(np.float64)

        # 2. Iteration Loop
        for _ in range(self.max_iters):
            # Vectorized distance computation (broadcasting): shape (N, K)
            distances = np.linalg.norm(X[:, None, :] - self.centroids[None, :, :], axis=2)
            labels = np.argmin(distances, axis=1)

            # Centroid update using list comprehension
            new_centroids = np.array([
                np.mean(X[labels == k_idx], axis=0) if np.any(labels == k_idx) else self.centroids[k_idx]
                for k_idx in range(self.k)
            ])

            # Convergence check: maximum centroid shift < tol
            shift = np.max(np.linalg.norm(new_centroids - self.centroids, axis=1))
            self.centroids = new_centroids
            if shift < self.tol:
                break

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Assign samples to closest cluster centroid."""
        distances = np.linalg.norm(X[:, None, :] - self.centroids[None, :, :], axis=2)
        return np.argmin(distances, axis=1)


In [ ]:
# --- Verification & Unit Test: K-Means Clustering ---
np.random.seed(42)

# Generate 3 distinct 2D clusters with 100 points each
c1 = np.random.randn(100, 2) * 0.5 + np.array([-5.0, -5.0])
c2 = np.random.randn(100, 2) * 0.5 + np.array([0.0, 5.0])
c3 = np.random.randn(100, 2) * 0.5 + np.array([5.0, -5.0])
X_clusters = np.vstack([c1, c2, c3])

kmeans = KMeans(k=3)
kmeans.fit(X_clusters)

labels = kmeans.predict(X_clusters)

print(f"K-Means fitted centroids:\n{kmeans.centroids}")

unique, counts = np.unique(labels, return_counts=True)
print(f"Cluster assignments count: {dict(zip(unique, counts))}")

assert len(unique) == 3, f"Expected 3 clusters, but found {len(unique)}"
assert all(85 <= c <= 115 for c in counts), "Cluster counts indicate poor convergence!"
print("✅ K-Means passed all verification checks!")


---
## 4. K-Nearest Neighbors (KNN)

KNN is a non-parametric, instance-based learning algorithm used for classification and regression.

### Mathematical Formulation
Given test query $x$ and training points $\{x_i, y_i\}_{i=1}^N$:
1. Compute Euclidean distance:
   $$
   d(x, x_i) = \|x - x_i\|_2 = \sqrt{\sum_{j=1}^D (x_j - x_{i,j})^2}
   $$
2. Identify indices of $K$ smallest distances: $\mathcal{N}_K(x)$.
3. **Classification:** Output majority vote $\hat{y} = \text{mode}(\{y_i : i \in \mathcal{N}_K(x)\})$.
4. **Regression:** Output mean $\hat{y} = \frac{1}{K} \sum_{i \in \mathcal{N}_K(x)} y_i$.

> **💡 Interview Gotchas & Pro-Tips:**
> - **Unified Predict Function:** Distance computation, $O(N)$ top-k partition, and voting are executed inside a single `predict()` function without separate helper methods.
> - **Top-K Complexity:**
>   - `np.argsort(distances)` takes $\mathcal{O}(N \log N)$.
>   - `np.argpartition(distances, kth=k)` takes $\mathcal{O}(N)$ average time. Always use `argpartition` for optimal performance.
> - **Vectorized Broadcasting:** Pairwise distances between $M$ test points and $N$ training points are computed in one line:
>   `distances = np.linalg.norm(X_test[:, None, :] - self.X_train[None, :, :], axis=2)`


In [ ]:
class KNearestNeighbors:
    """
    Vectorized K-Nearest Neighbors in a clean, self-contained implementation.
    Computes pairwise distances, O(N) top-K partition, and voting inside predict().
    """
    def __init__(self, k: int = 5, task: str = "classification"):
        assert task in ("classification", "regression"), "task must be 'classification' or 'regression'"
        self.k = k
        self.task = task
        self.X_train = None
        self.y_train = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "KNearestNeighbors":
        """Lazy learner: store training data."""
        self.X_train = np.asarray(X, dtype=np.float64)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X_test: np.ndarray) -> np.ndarray:
        """
        Unified prediction function:
        1. Computes (M, N) pairwise distance matrix via broadcasting.
        2. Retrieves top-k nearest neighbors via O(N) argpartition.
        3. Computes mode (classification) or mean (regression).
        """
        X_test = np.asarray(X_test, dtype=np.float64)
        M = X_test.shape[0]

        # Vectorized pairwise Euclidean distances: (M, 1, D) - (1, N, D) -> (M, N)
        distances = np.linalg.norm(X_test[:, None, :] - self.X_train[None, :, :], axis=2)

        predictions = []
        for i in range(M):
            row_dists = distances[i]
            # O(N) selection of k nearest neighbor indices
            k_indices = np.argpartition(row_dists, kth=self.k - 1)[:self.k]
            k_labels = self.y_train[k_indices]

            if self.task == "classification":
                values, counts = np.unique(k_labels, return_counts=True)
                pred = values[np.argmax(counts)]
            else:
                pred = np.mean(k_labels)

            predictions.append(pred)

        return np.array(predictions)


In [ ]:
# --- Verification & Unit Test: KNN ---
np.random.seed(42)
N_train = 150

# Generate 3 training classes
X_tr = np.vstack([
    np.random.randn(N_train // 3, 2) + np.array([-3.0, 0.0]),
    np.random.randn(N_train // 3, 2) + np.array([3.0, 0.0]),
    np.random.randn(N_train // 3, 2) + np.array([0.0, 4.0])
])
y_tr = np.array([0] * (N_train // 3) + [1] * (N_train // 3) + [2] * (N_train // 3))

# Query test points close to each center
X_te = np.array([[-3.0, 0.0], [3.0, 0.0], [0.0, 4.0]])
expected_y = np.array([0, 1, 2])

knn = KNearestNeighbors(k=5, task="classification")
knn.fit(X_tr, y_tr)
preds = knn.predict(X_te)

print(f"KNN Test predictions: {preds}, Expected: {expected_y}")
assert np.array_equal(preds, expected_y), f"Expected {expected_y}, got {preds}"
print("✅ K-Nearest Neighbors passed all verification checks!")


---
## 5. 2-Layer Multi-Layer Perceptron (Backprop from Scratch in NumPy)

Implementing an MLP from scratch using pure NumPy proves complete mastery over the **multivariate chain rule**, activation derivatives, and tensor shapes.

### Architecture & Forward Pass
- **Input:** $X \in \mathbb{R}^{N \times D_{\text{in}}}$
- **Hidden Layer:**
  $$
  Z_1 = X W_1 + b_1 \in \mathbb{R}^{N \times H}, \qquad A_1 = \text{ReLU}(Z_1) = \max(0, Z_1)
  $$
- **Output Layer:**
  $$
  Z_2 = A_1 W_2 + b_2 \in \mathbb{R}^{N \times 1}, \qquad \hat{Y} = \sigma(Z_2) \in \mathbb{R}^{N \times 1}
  $$
- **Loss:** Binary Cross-Entropy
  $$
  \mathcal{L} = -\frac{1}{N} \sum_{i=1}^N \left[ Y_i \log(\hat{Y}_i) + (1 - Y_i)\log(1 - \hat{Y}_i) \right]
  $$

### Backward Pass (Analytical Chain Rule Derivation)
1. **Output layer gradient:**
   $$
   \delta_2 = \frac{\partial \mathcal{L}}{\partial Z_2} = \frac{1}{N} (\hat{Y} - Y) \in \mathbb{R}^{N \times 1}
   $$
2. **Output weights and bias gradients:**
   $$
   \frac{\partial \mathcal{L}}{\partial W_2} = A_1^T \delta_2 \in \mathbb{R}^{H \times 1}, \qquad \frac{\partial \mathcal{L}}{\partial b_2} = \sum_{i=1}^N \delta_{2, i} \in \mathbb{R}^{1 \times 1}
   $$
3. **Hidden layer activation gradient:**
   $$
   \frac{\partial \mathcal{L}}{\partial A_1} = \delta_2 W_2^T \in \mathbb{R}^{N \times H}
   $$
4. **Hidden pre-activation gradient:**
   $$
   \delta_1 = \frac{\partial \mathcal{L}}{\partial Z_1} = \left(\delta_2 W_2^T\right) \odot \mathbf{1}_{\{Z_1 > 0\}} \in \mathbb{R}^{N \times H}
   $$
5. **Input weights and bias gradients:**
   $$
   \frac{\partial \mathcal{L}}{\partial W_1} = X^T \delta_1 \in \mathbb{R}^{D_{\text{in}} \times H}, \qquad \frac{\partial \mathcal{L}}{\partial b_1} = \sum_{i=1}^N \delta_{1, i} \in \mathbb{R}^{1 \times H}
   $$

> **💡 Interview Gotchas & Pro-Tips:**
> - **Symmetry Breaking:** Initializing weights to all zeros means every hidden neuron computes identical outputs and receives identical gradients. Always use **He (Kaiming) Normal** initialization: $W \sim \mathcal{N}(0, \sqrt{2 / D_{\text{in}}})$.
> - **Shape Matching:** Always verify shapes at each stage:
>   `X: (N, D) -> Z1: (N, H) -> Z2: (N, 1) -> dW2: (H, 1) -> dW1: (D, H)`.


In [ ]:
class TwoLayerMLP:
    """
    2-Layer Multi-Layer Perceptron with manual backpropagation implemented in pure NumPy.
    Architecture: Input -> Linear -> ReLU -> Linear -> Sigmoid (BCE Loss).
    """
    def __init__(self, in_dim: int, hidden_dim: int, lr: float = 0.1):
        self.lr = lr
        
        # He (Kaiming) Normal Initialization for ReLU
        self.W1 = np.random.randn(in_dim, hidden_dim) * np.sqrt(2.0 / in_dim)
        self.b1 = np.zeros((1, hidden_dim))
        
        # Xavier Normal Initialization for Sigmoid output
        self.W2 = np.random.randn(hidden_dim, 1) * np.sqrt(1.0 / hidden_dim)
        self.b2 = np.zeros((1, 1))
        
        self.loss_history = []

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))

    def forward(self, X: np.ndarray):
        """
        Forward pass. Returns final predictions and intermediate cache for backprop.
        """
        Z1 = np.dot(X, self.W1) + self.b1           # Shape: (N, H)
        A1 = np.maximum(0.0, Z1)                     # ReLU activation
        
        Z2 = np.dot(A1, self.W2) + self.b2          # Shape: (N, 1)
        A2 = self._sigmoid(Z2)                      # Sigmoid output
        
        cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
        return A2, cache

    def backward(self, Y: np.ndarray, cache: dict):
        """
        Backward pass: computes exact analytical gradients via multivariate chain rule.
        """
        N = Y.shape[0]
        X = cache["X"]
        Z1 = cache["Z1"]
        A1 = cache["A1"]
        A2 = cache["A2"]
        
        # 1. Output gradient: dL/dZ2 = (1/N) * (A2 - Y)
        dZ2 = (1.0 / N) * (A2 - Y)                  # Shape: (N, 1)
        dW2 = np.dot(A1.T, dZ2)                     # Shape: (H, 1)
        db2 = np.sum(dZ2, axis=0, keepdims=True)    # Shape: (1, 1)
        
        # 2. Backprop to hidden layer: dL/dZ1 = (dZ2 @ W2.T) * ReLU'(Z1)
        dA1 = np.dot(dZ2, self.W2.T)                # Shape: (N, H)
        dZ1 = dA1 * (Z1 > 0).astype(float)          # ReLU derivative
        
        dW1 = np.dot(X.T, dZ1)                      # Shape: (D, H)
        db1 = np.sum(dZ1, axis=0, keepdims=True)    # Shape: (1, H)
        
        # 3. Parameter updates (Gradient Descent)
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 2000):
        N = X.shape[0]
        Y = y.reshape(-1, 1)
        eps = 1e-15
        
        for epoch in range(epochs):
            # Forward pass
            preds, cache = self.forward(X)
            
            # Loss computation
            preds_clipped = np.clip(preds, eps, 1.0 - eps)
            loss = - (1.0 / N) * np.sum(Y * np.log(preds_clipped) + (1.0 - Y) * np.log(1.0 - preds_clipped))
            self.loss_history.append(loss)
            
            # Backward pass
            self.backward(Y, cache)
            
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        preds, _ = self.forward(X)
        return (preds >= 0.5).astype(int).squeeze()


In [ ]:
# --- Verification & Unit Test: 2-Layer MLP on Non-Linear XOR Problem ---
np.random.seed(42)

# XOR problem: non-linear classification boundary
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float64)
y_xor = np.array([0, 1, 1, 0], dtype=np.float64)

X_train = np.repeat(X_xor, 50, axis=0) + np.random.randn(200, 2) * 0.05
y_train = np.repeat(y_xor, 50)

mlp = TwoLayerMLP(in_dim=2, hidden_dim=8, lr=0.5)
mlp.fit(X_train, y_train, epochs=2500)

xor_preds = mlp.predict(X_xor)
print("XOR Truth:", y_xor)
print("XOR Preds:", xor_preds)
print(f"MLP Initial Loss: {mlp.loss_history[0]:.4f} -> Final Loss: {mlp.loss_history[-1]:.4f}")

assert np.array_equal(xor_preds, y_xor), "MLP failed to solve non-linear XOR problem!"
assert mlp.loss_history[-1] < 0.1, "Loss did not converge!"
print("✅ 2-Layer MLP passed all verification checks!")


---
## 6. Layer Normalization (LayerNorm)

Introduced by Ba, Kiros, & Hinton (2016), Layer Normalization is an indispensable building block of modern Transformers (GPT, BERT, LLaMA).

### Mathematical Formulation
Given an input tensor $x \in \mathbb{R}^{B \times T \times D}$ (or $x \in \mathbb{R}^{N \times D}$):
For each individual sample and token position, we compute mean $\mu$ and variance $\sigma^2$ across the **last dimension** (feature dimension $D$):

$$
\mu = \frac{1}{D} \sum_{j=1}^D x_j, \qquad \sigma^2 = \frac{1}{D} \sum_{j=1}^D (x_j - \mu)^2
$$

Normalize and apply learnable affine parameters $\gamma$ (scale) and $\beta$ (shift):
$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$
$$
y = \gamma \odot \hat{x} + \beta
$$

Where:
- $\gamma \in \mathbb{R}^D$ is initialized to $\mathbf{1}$.
- $\beta \in \mathbb{R}^D$ is initialized to $\mathbf{0}$.
- $\epsilon > 0$ is a small scalar (e.g. $10^{-5}$) to prevent division by zero.

---

### 🆚 Critical Interview Comparison: LayerNorm vs. BatchNorm

| Feature | **BatchNorm** | **LayerNorm** |
|---|---|---|
| **Dimension normalized across** | Across the **batch dimension** $B$ for each feature channel | Across the **feature dimension** $D$ for each sample independently |
| **Dependency between batch samples** | **Yes** (a sample's output depends on other batch members) | **No** (each sample is normalized completely independently) |
| **Behavior during Inference** | Requires tracking running stats (`running_mean`, `running_var`) | Exact same computation during train and inference! |
| **Suitability for Transformers & NLP**| Poor (fails with variable sequence lengths & small batch sizes) | **Ideal** (standard for LLMs and recurrent models) |


In [ ]:
# --- Pure NumPy Implementation of LayerNorm ---
def layer_norm_numpy(x: np.ndarray, gamma: np.ndarray, beta: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    """
    x: shape (..., D)
    gamma: shape (D,)
    beta: shape (D,)
    """
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_hat = (x - mean) / np.sqrt(var + eps)
    return gamma * x_hat + beta

# --- PyTorch Module Implementation of LayerNorm ---
try:
    import torch
    import torch.nn as nn

    class CustomLayerNorm(nn.Module):
        def __init__(self, normalized_shape: int, eps: float = 1e-5):
            super().__init__()
            self.eps = eps
            # Learnable parameters: gamma (initialized to 1), beta (initialized to 0)
            self.gamma = nn.Parameter(torch.ones(normalized_shape))
            self.beta = nn.Parameter(torch.zeros(normalized_shape))

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            # x shape: (..., D)
            mean = x.mean(dim=-1, keepdim=True)
            var = x.var(dim=-1, unbiased=False, keepdim=True)  # MLE variance (divide by D, not D-1)
            x_norm = (x - mean) / torch.sqrt(var + self.eps)
            return self.gamma * x_norm + self.beta

    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False


In [ ]:
# --- Verification: LayerNorm Numerical Accuracy ---
# 1. Test NumPy LayerNorm
x_np = np.array([[1.0, 2.0, 3.0, 4.0], [10.0, 20.0, 30.0, 40.0]])
gamma_np = np.ones(4)
beta_np = np.zeros(4)

out_np = layer_norm_numpy(x_np, gamma_np, beta_np)
print("NumPy LayerNorm output shape:", out_np.shape)
assert np.allclose(np.mean(out_np, axis=-1), 0.0, atol=1e-6)
assert np.allclose(np.var(out_np, axis=-1), 1.0, atol=1e-3)
print("NumPy LayerNorm normalized correctly to mean 0 and var 1.")

# 2. Test PyTorch LayerNorm vs official torch.nn.LayerNorm
if TORCH_AVAILABLE:
    B, T, D = 2, 4, 8
    x_torch = torch.randn(B, T, D)
    
    custom_ln = CustomLayerNorm(D)
    official_ln = nn.LayerNorm(D, eps=1e-5)
    
    with torch.no_grad():
        official_ln.weight.copy_(custom_ln.gamma)
        official_ln.bias.copy_(custom_ln.beta)
        
    custom_out = custom_ln(x_torch)
    official_out = official_ln(x_torch)
    
    assert torch.allclose(custom_out, official_out, atol=1e-5), "Custom LayerNorm did not match official PyTorch!"
    print("✅ PyTorch CustomLayerNorm matched official torch.nn.LayerNorm within 1e-5 precision!")
else:
    print("⚠️ PyTorch not installed in this environment; NumPy LayerNorm verified.")


---
## 7. Multi-Head Self-Attention (Transformer Core)

Multi-Head Attention (Vaswani et al., 2017) is the central engine of the Transformer. It enables the model to simultaneously attend to information from different representation subspaces at different positions.

### Mathematical Formulation
Given sequence $X \in \mathbb{R}^{B \times T \times D_{\text{model}}}$:
1. **Linear Projections:** Project into Query ($Q$), Key ($K$), and Value ($V$):
   $$
   Q = X W_Q, \qquad K = X W_K, \qquad V = X W_V
   $$
2. **Multi-Head Reshape:** Split $D_{\text{model}}$ into $H$ heads each of dimension $d_k = D / H$:
   $$
   \text{Shape: } (B, T, D) \to (B, H, T, d_k)
   $$
3. **Scaled Dot-Product Attention:**
   $$
   \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}} + M\right) V
   $$
   where $M$ is an optional causal mask ($M_{i,j} = -\infty$ for $j > i$).
4. **Concatenate & Output Projection:**
   Combine the $H$ heads back to $(B, T, D)$ and project with $W_O \in \mathbb{R}^{D \times D}$.

```
Input (B, T, D)
  ├──> W_Q ──> Q (B, H, T, d_k) ─┐
  ├──> W_K ──> K (B, H, T, d_k) ──┴──> (Q @ K^T) / sqrt(d_k) + Mask ──> Softmax ──┐
  └──> W_V ──> V (B, H, T, d_k) ──────────────────────────────────────────────────┴──> Attention (B, H, T, d_k)
                                                                                          │
                                                      Reshape & Concat <──────────────────┘
                                                            │
                                                          W_O ──> Output (B, T, D)
```

> **💡 Interview Gotchas & Pro-Tips:**
> - **Why scale by $\frac{1}{\sqrt{d_k}}$?** For large $d_k$, the dot product $\sum_{i=1}^{d_k} q_i k_i$ has variance equal to $d_k$. Without scaling, the values explode in magnitude, pushing softmax into regions with near-zero gradients (vanishing gradient problem).
> - **Causal Mask Implementation:** Use `torch.tril(torch.ones(T, T))` and `.masked_fill(mask == 0, float('-inf'))`.
> - **Efficient Projection:** Instead of creating 3 separate `nn.Linear` layers, top implementations use a single `nn.Linear(D, 3 * D)` and call `.chunk(3, dim=-1)`.


In [ ]:
if TORCH_AVAILABLE:
    import torch.nn.functional as F

    class MultiHeadSelfAttention(nn.Module):
        """
        Production-quality Multi-Head Attention module from scratch.
        Supports causal masking (autoregressive decoding) and batched computation.
        """
        def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
            super().__init__()
            assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
            
            self.d_model = d_model
            self.n_heads = n_heads
            self.d_k = d_model // n_heads
            
            # Single combined projection for Q, K, V (3 * d_model) for optimal GPU efficiency
            self.qkv_proj = nn.Linear(d_model, 3 * d_model)
            self.out_proj = nn.Linear(d_model, d_model)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x: torch.Tensor, is_causal: bool = False) -> torch.Tensor:
            """
            x: Input tensor of shape (B, T, D)
            is_causal: If True, applies lower-triangular causal attention mask
            """
            B, T, D = x.shape
            
            # 1. Linear projection -> (B, T, 3 * D)
            qkv = self.qkv_proj(x)
            
            # Split into Q, K, V -> each (B, T, D)
            q, k, v = qkv.chunk(3, dim=-1)
            
            # 2. Reshape and transpose for multi-head: (B, T, H, d_k) -> (B, H, T, d_k)
            q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
            
            # 3. Scaled dot-product: (B, H, T, d_k) @ (B, H, d_k, T) -> (B, H, T, T)
            scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
            
            # 4. Apply causal mask if required
            if is_causal:
                causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
                scores = scores.masked_fill(~causal_mask, float("-inf"))
                
            # 5. Softmax along key sequence dimension
            attn_weights = F.softmax(scores, dim=-1)
            attn_weights = self.dropout(attn_weights)
            
            # 6. Weighted sum over values: (B, H, T, T) @ (B, H, T, d_k) -> (B, H, T, d_k)
            context = torch.matmul(attn_weights, v)
            
            # 7. Concatenate heads: (B, H, T, d_k) -> (B, T, H, d_k) -> (B, T, D)
            context = context.transpose(1, 2).contiguous().view(B, T, D)
            
            # 8. Final linear projection
            return self.out_proj(context)


In [ ]:
# --- Verification: Multi-Head Self-Attention ---
if TORCH_AVAILABLE:
    torch.manual_seed(42)
    B, T, D, H = 2, 6, 64, 4
    x = torch.randn(B, T, D)
    
    mha = MultiHeadSelfAttention(d_model=D, n_heads=H)
    
    # Test bidirectional attention
    out_bidi = mha(x, is_causal=False)
    assert out_bidi.shape == (B, T, D), f"Expected {(B, T, D)}, got {out_bidi.shape}"
    
    # Test causal attention
    out_causal = mha(x, is_causal=True)
    assert out_causal.shape == (B, T, D), f"Expected {(B, T, D)}, got {out_causal.shape}"
    
    # Test that causal mask prevents future tokens from affecting past tokens
    x_perturbed = x.clone()
    x_perturbed[:, -1, :] += 10.0
    out_perturbed_causal = mha(x_perturbed, is_causal=True)
    
    token_0_diff = (out_causal[:, 0, :] - out_perturbed_causal[:, 0, :]).abs().max().item()
    assert token_0_diff < 1e-6, f"Causal mask failed! Future token affected position 0: diff={token_0_diff}"
    
    print(f"MHA Output Shape: {out_causal.shape}")
    print("✅ Multi-Head Self-Attention passed all verification & causality checks!")
else:
    print("⚠️ PyTorch not installed; skipped MHA execution.")


---
## 8. Low-Rank Adaptation (LoRA) Layer

LoRA (Hu et al., 2021) is the standard parameter-efficient fine-tuning (PEFT) method used to adapt large foundation models (LLaMA, Mistral) by training $< 1\%$ of total parameters.

### Mathematical Formulation
For a pre-trained linear layer $h = W_0 x$, LoRA decomposes the weight update $\Delta W$ into two low-rank matrices:
$$
h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} (B A) x
$$

Where:
- $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$ is **frozen** (`requires_grad = False`).
- $A \in \mathbb{R}^{r \times d_{\text{in}}}$: down-projection with rank $r \ll \min(d_{\text{in}}, d_{\text{out}})$.
- $B \in \mathbb{R}^{d_{\text{out}} \times r}$: up-projection.
- $\alpha$: constant scaling hyperparameter.
- $\frac{\alpha}{r}$: scaling factor to keep learning dynamics stable across different rank choices.

```
       Input x (..., d_in)
         ├───> [Frozen W_0] (d_out x d_in) ─────────────────────┐
         │                                                      │ (+) ──> Output (..., d_out)
         └───> [Trainable A] (r x d_in) ──> [Trainable B] (d_out x r) * (alpha / r) ──┘
```

> **💡 The Two Critical Interview Traps of LoRA:**
> 1. **The Initialization Trap:**
>    - Matrix $A$ is initialized with random Gaussian / Kaiming distribution.
>    - Matrix $B$ **MUST be initialized to exact zeros** (`nn.init.zeros_(self.lora_B)`).
>    - **Why?** Because at step 0, $\Delta W = B \times A = 0 \times A = 0$. This guarantees that before training begins, the adapted model behaves **identically** to the base model with zero modification!
> 2. **Zero-Inference Overhead (Weight Folding):**
>    - During deployment, compute $W_{\text{merged}} = W_0 + \frac{\alpha}{r} B A$.
>    - LoRA adds **zero latency** and **zero extra FLOPs** during production inference!


In [ ]:
if TORCH_AVAILABLE:
    import math

    class LoRALinear(nn.Module):
        """
        Low-Rank Adaptation (LoRA) Linear Layer wrapper from scratch.
        """
        def __init__(self, in_features: int, out_features: int, rank: int = 8, alpha: float = 16.0):
            super().__init__()
            self.in_features = in_features
            self.out_features = out_features
            self.rank = rank
            self.scaling = alpha / rank
            self.merged = False

            # Base frozen weights
            self.weight = nn.Parameter(torch.randn(out_features, in_features), requires_grad=False)
            self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)

            # Low-rank adapter matrices
            self.lora_A = nn.Parameter(torch.empty(rank, in_features))
            self.lora_B = nn.Parameter(torch.empty(out_features, rank))

            self.reset_parameters()

        def reset_parameters(self):
            # 1. Kaiming uniform for A
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            # 2. Crucial: B initialized to EXACT ZERO so delta_W starts at 0!
            nn.init.zeros_(self.lora_B)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            if self.merged:
                return F.linear(x, self.weight, self.bias)
            
            # Base linear pass: x @ W_0^T + b
            base_out = F.linear(x, self.weight, self.bias)
            
            # LoRA branch: (x @ A^T) @ B^T * scaling
            lora_out = (x @ self.lora_A.T) @ self.lora_B.T * self.scaling
            return base_out + lora_out

        def merge_weights(self):
            """Fold LoRA delta into base weight for zero-latency inference."""
            if not self.merged:
                delta_w = (self.lora_B @ self.lora_A) * self.scaling
                self.weight.data += delta_w
                self.merged = True

        def unmerge_weights(self):
            """Unfold LoRA weights back to continue fine-tuning."""
            if self.merged:
                delta_w = (self.lora_B @ self.lora_A) * self.scaling
                self.weight.data -= delta_w
                self.merged = False


In [ ]:
# --- Verification: LoRA Layer ---
if TORCH_AVAILABLE:
    torch.manual_seed(42)
    in_dim, out_dim, rank = 128, 64, 4
    x_test = torch.randn(8, in_dim)
    
    lora_layer = LoRALinear(in_features=in_dim, out_features=out_dim, rank=rank, alpha=8.0)
    
    # 1. Check parameter counts
    trainable_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
    frozen_params = sum(p.numel() for p in lora_layer.parameters() if not p.requires_grad)
    print(f"Frozen parameters (W_0, bias): {frozen_params}")
    print(f"Trainable LoRA parameters (A, B): {trainable_params} ({trainable_params / (frozen_params + trainable_params) * 100:.2f}%)")
    
    # 2. Check Initialization Trap: initial output must EQUAL base linear output exactly
    base_linear_out = F.linear(x_test, lora_layer.weight, lora_layer.bias)
    initial_lora_out = lora_layer(x_test)
    assert torch.allclose(initial_lora_out, base_linear_out), "LoRA initialization trap violated! Output != base output at step 0"
    
    # 3. Simulate training update: modify lora_B
    with torch.no_grad():
        lora_layer.lora_B.add_(torch.randn_like(lora_layer.lora_B) * 0.1)
        
    out_before_merge = lora_layer(x_test)
    
    # 4. Test weight merging equivalence
    lora_layer.merge_weights()
    out_after_merge = lora_layer(x_test)
    assert torch.allclose(out_before_merge, out_after_merge, atol=1e-5), "Merged output does not match unmerged output!"
    
    print("✅ LoRA Layer passed parameter-efficiency, initialization trap, and weight-merging checks!")
else:
    print("⚠️ PyTorch not installed; skipped LoRA execution.")


---
## 9. Byte-Pair Encoding (BPE) Tokenizer

Byte-Pair Encoding (Gage 1994, Sennrich et al. 2015) is the subword tokenization algorithm powering modern LLMs (GPT-2, GPT-3, GPT-4, LLaMA). It provides the optimal trade-off between character-level (too long sequences) and word-level (out-of-vocabulary words).

### Algorithmic Steps
1. **Base Vocabulary:** Split each word in the training corpus into characters, appending an end-of-word marker `</w>`:
   `"low" -> ('l', 'o', 'w', '</w>')`
2. **Frequency Counting:** Count all adjacent symbol pairs across the entire dictionary:
   `('l', 'o') -> 5, ('o', 'w') -> 5, ('w', '</w>') -> 5`
3. **Merge Step:** Identify the most frequent pair $(c_1, c_2)$ and merge it into a single token $c_1c_2$:
   `('l', 'o', 'w', '</w>') -> ('lo', 'w', '</w>')`
4. **Iterate:** Repeat for $K$ merge steps or until target vocabulary size is reached.
5. **Inference (Tokenizing New Text):** Apply the learned merge rules **in the exact chronological order** they were learned.

```
Initial:  ['l', 'o', 'w', '</w>']  ['l', 'o', 'w', 'e', 's', 't', '</w>']
Merge 1:  ['lo', 'w', '</w>']      ['lo', 'w', 'e', 's', 't', '</w>']
Merge 2:  ['low', '</w>']          ['low', 'e', 's', 't', '</w>']
Merge 3:  ['low', '</w>']          ['low', 'est', '</w>']
```

> **💡 Interview Gotchas & Pro-Tips:**
> - **End-of-word marker `</w>`:** Crucial for distinguishing between substrings inside words vs at the ends (e.g. "th" at the start of "the" vs at the end of "with").
> - **Subword handling of unseen/OOD words:** If a test word has never been seen before, it gracefully decomposes into individual characters without throwing `<UNK>`!


In [ ]:
import collections
import re
from typing import Dict, Tuple, List

class BPETokenizer:
    """
    Byte-Pair Encoding (BPE) Tokenizer implemented from scratch in pure Python.
    """
    def __init__(self, num_merges: int = 10):
        self.num_merges = num_merges
        self.merges: List[Tuple[str, str]] = []
        self.vocab: List[str] = []

    @staticmethod
    def _get_stats(vocab: Dict[Tuple[str, ...], int]) -> Dict[Tuple[str, str], int]:
        """Count frequency of all adjacent symbol pairs across vocabulary."""
        pairs = collections.defaultdict(int)
        for word_tuple, freq in vocab.items():
            for i in range(len(word_tuple) - 1):
                pairs[(word_tuple[i], word_tuple[i + 1])] += freq
        return pairs

    @staticmethod
    def _merge_vocab(pair: Tuple[str, str], vocab: Dict[Tuple[str, ...], int]) -> Dict[Tuple[str, ...], int]:
        """Merge all occurrences of pair in vocabulary words."""
        v_out = {}
        first, second = pair
        new_token = first + second
        
        for word_tuple, freq in vocab.items():
            new_word = []
            i = 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and word_tuple[i] == first and word_tuple[i + 1] == second:
                    new_word.append(new_token)
                    i += 2
                else:
                    new_word.append(word_tuple[i])
                    i += 1
            v_out[tuple(new_word)] = freq
        return v_out

    def fit(self, corpus: str) -> "BPETokenizer":
        words = re.findall(r"\S+", corpus)
        word_freqs = collections.Counter(words)
        
        vocab = {tuple(list(word) + ["</w>"]): freq for word, freq in word_freqs.items()}
        
        self.merges = []
        for step in range(self.num_merges):
            pairs = self._get_stats(vocab)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            vocab = self._merge_vocab(best_pair, vocab)
            self.merges.append(best_pair)
            
        vocab_set = set()
        for word_tuple in vocab.keys():
            for token in word_tuple:
                vocab_set.add(token)
        self.vocab = sorted(list(vocab_set))
        return self

    def tokenize(self, text: str) -> List[str]:
        """Tokenize an unseen input string using the learned merge rules in chronological order."""
        words = re.findall(r"\S+", text)
        tokens = []
        
        for word in words:
            word_tuple = tuple(list(word) + ["</w>"])
            
            for pair in self.merges:
                first, second = pair
                new_token = first + second
                new_word = []
                i = 0
                while i < len(word_tuple):
                    if i < len(word_tuple) - 1 and word_tuple[i] == first and word_tuple[i + 1] == second:
                        new_word.append(new_token)
                        i += 2
                    else:
                        new_word.append(word_tuple[i])
                        i += 1
                word_tuple = tuple(new_word)
                
            tokens.extend(list(word_tuple))
            
        return tokens


In [ ]:
# --- Verification & Unit Test: BPE Tokenizer ---
corpus = "low low low low low lower lower newest newest widest widest widest"
bpe = BPETokenizer(num_merges=6)
bpe.fit(corpus)

print("Top 6 Learned BPE Merges in order:")
for i, merge in enumerate(bpe.merges, 1):
    print(f"  {i}. {merge} -> '{merge[0] + merge[1]}'")

test_sentence = "lower lowest"
tokens = bpe.tokenize(test_sentence)
print(f"\nTokenized '{test_sentence}':\n  {tokens}")

assert len(bpe.merges) == 6, "Expected 6 merges to be recorded"
assert any("low" in t for t in tokens), "Expected 'low' subword in tokenization"
print("✅ Byte-Pair Encoding passed all verification checks!")


---
## 10. MLE Interview Quick Reference & Cheat Sheet

### Time & Space Complexity Summary
| Algorithm | Train Time | Inference Time | Space Complexity |
|---|---|---|---|
| **Linear Reg (GD)** | $\mathcal{O}(\text{iters} \cdot N D)$ | $\mathcal{O}(D)$ | $\mathcal{O}(D)$ |
| **Linear Reg (Normal Eq)** | $\mathcal{O}(D^3 + N D^2)$ | $\mathcal{O}(D)$ | $\mathcal{O}(D^2)$ |
| **Logistic Regression** | $\mathcal{O}(\text{iters} \cdot N D)$ | $\mathcal{O}(D)$ | $\mathcal{O}(D)$ |
| **K-Means** | $\mathcal{O}(\text{iters} \cdot N K D)$ | $\mathcal{O}(K D)$ | $\mathcal{O}(N D + K D)$ |
| **KNN** | $\mathcal{O}(1)$ (Lazy) | $\mathcal{O}(N D + N)$ | $\mathcal{O}(N D)$ |
| **Multi-Head Attention** | $\mathcal{O}(T^2 D)$ | $\mathcal{O}(T^2 D)$ (or $\mathcal{O}(T D)$ w/ KV Cache) | $\mathcal{O}(B T D + B H T^2)$ |
| **LayerNorm** | $\mathcal{O}(N D)$ | $\mathcal{O}(N D)$ | $\mathcal{O}(D)$ |
| **LoRA** | $\mathcal{O}(r (d_{\text{in}} + d_{\text{out}}))$ | $\mathcal{O}(1)$ after fold | $\mathcal{O}(r (d_{\text{in}} + d_{\text{out}}))$ |

---

### Top Whiteboard Traps Checklist
1. **Linear Reg:** Check for collinearity $\to$ recommend `np.linalg.pinv`.
2. **Logistic Reg:** Watch out for `np.log(0)` $\to$ use `np.clip` and branch sigmoid for $z \ge 0$ vs $z < 0$.
3. **K-Means:** Never use nested loops for distances $\to$ use NumPy broadcasting. Initialize via random choice without replacement and update centroids via list comprehension.
4. **KNN:** Don't use `np.argsort` ($\mathcal{O}(N \log N)$) $\to$ use `np.argpartition` ($\mathcal{O}(N)$).
5. **Backpropagation:** Remember He initialization for ReLU to break symmetry.
6. **LayerNorm:** Normalize over the **feature dimension** $D$, NOT the batch dimension!
7. **Multi-Head Attention:** Scale dot products by $\frac{1}{\sqrt{d_k}}$ to avoid vanishing gradients in softmax.
8. **LoRA:** Matrix $B$ **must be initialized to zero** so initial adaptation is zero.
9. **BPE:** Always preserve the order of learned merge rules at inference time.
